# **Maskininlärning (ML) är avgörande för att identifiera avvikande beteenden i nätverk, särskilt mot DDoS-attacker, intrångsförsök och zero-day-exploits. Här är de bästa ML-modellerna för detta ändamål.**

# **0. Generera data**

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

# Skapa simulerad nätverksdata
np.random.seed(42)
num_samples = 1000

data = {
    'duration': np.random.exponential(scale=1.0, size=num_samples),
    'src_bytes': np.random.randint(0, 5000, size=num_samples),
    'dst_bytes': np.random.randint(0, 5000, size=num_samples),
    'count': np.random.poisson(lam=5, size=num_samples),
    'same_srv_rate': np.random.uniform(0, 1, size=num_samples),
    'dst_host_count': np.random.poisson(lam=10, size=num_samples),
    'dst_host_srv_count': np.random.poisson(lam=7, size=num_samples),
    # Label: 0=normal, 1=attack (cirka 30% attacker)
    'label': np.random.choice([0, 1], size=num_samples, p=[0.7, 0.3])
}

# Skapa DataFrame
df = pd.DataFrame(data)

# Spara som CSV
df.to_csv('network_data.csv', index=False)

# Ladda ner filen om du vill använda den senare
files.download('network_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **1. Övervakad Inlärning (Supervised Learning)**
**Används när: Historisk data med etiketter ("normal" vs "attack") finns.
Bäst för: Klassificering av kända hot.**

**A. Random Forest**


Hanterar stora datamängder med högre noggrannhet.

Motståndskraftig mot överanpassning.

Bra för både numeriska och kategoriska data.

In [1]:
# Enkel övervakad inlärning för nätverkssäkerhet

# Steg 1: Importera bibliotek
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# Steg 2: Ladda upp datafil
print("Ladda upp 'network_data.csv':")
uploaded = files.upload()
data = pd.read_csv(next(iter(uploaded)))  # Läser in filen

# Steg 3: Visa enkel information om datan
print("\nFörhandsvisning av datan:")
print(data.head())
print(data.describe())

print("\nAntal normala (0) och attacker (1):")
print(data['label'].value_counts())

# Steg 4: Dela upp i funktioner (X) och etiketter (y)
X = data.drop("label", axis=1)
y = data["label"]

# Steg 5: Dela upp i träningsdata och testdata
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Steg 6: Träna modellen
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Steg 7: Utvärdera modellen
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModellens träffsäkerhet: {accuracy:.2%}")

cm = confusion_matrix(y_test, y_pred)
print("\nFörvirringsmatris:")
print(f"Normal gissad som normal:     {cm[0][0]}")
print(f"Normal gissad som attack:     {cm[0][1]}")
print(f"Attack gissad som normal:     {cm[1][0]}")
print(f"Attack gissad som attack:     {cm[1][1]}")

# Steg 9: Kort förklaring
print("\nRandom Forest är en modell som:")
print("- Kombinerar många beslutsträd")
print("- Fungerar bra med olika typer av data")
print("- Ger stabila resultat även vid brusig data")


Ladda upp 'network_data.csv':


Saving network_data.csv to network_data.csv

Förhandsvisning av datan:
   duration  src_bytes  dst_bytes  count  same_srv_rate  dst_host_count  \
0  0.469268       4107       2037      4       0.274413              13   
1  3.010121       1341       3423      4       0.393170               5   
2  1.316746       3663       2567      8       0.387764               3   
3  0.912943        215       1573      6       0.286867               8   
4  0.169625        338       3694      6       0.268580              12   

   dst_host_srv_count  label  
0                  10      1  
1                   4      0  
2                   8      0  
3                   8      0  
4                   9      0  
          duration    src_bytes    dst_bytes        count  same_srv_rate  \
count  1000.000000  1000.000000  1000.000000  1000.000000    1000.000000   
mean      0.972506  2463.576000  2490.097000     4.925000       0.493825   
std       0.972510  1454.884751  1411.913589     2.099472       

**Förhandsvisningen** ger oss en snabb översikt av datan så att vi kan:

1. Förstå datans struktur.

2. Se om vi har rätt typ av data för att bygga en modell.

3. Kontrollera om det finns några uppenbara problem i datan, som null-värden eller felaktiga format.

Varje rad representerar ett nätverksflöde med funktioner som duration (tid), src_bytes (bytes skickade från källan), dst_bytes (bytes skickade till destinationen), osv.

label är den kolumn som säger om trafikflödet är "normal" (0) eller en "attack" (1).


🔍 Datan visar nätverkshändelser, där varje rad är en anslutning.

label = 0 betyder normal trafik

label = 1 betyder attack

📊 Totalt:

699 normala händelser

301 attacker

✅ **Modellen har 69 % träffsäkerhet, alltså den gissar rätt i 69 av 100 fall.**

⚠️ Men eftersom de flesta händelser är normala, kan modellen vara dålig på att hitta attacker — även om träffsäkerheten verkar okej.

📌 **Tolkning:**

✅ 197 normala händelser gissades rätt.

❌ 78 attacker missades (gissades som normal trafik).

⚠️ Endast 10 attacker upptäcktes korrekt.

🔔 15 normala händelser felklassades som attacker (falska alarm).

⚠️ Viktig insikt:
Modellen är bra på att känna igen normal trafik, men missar de flesta attacker. Det betyder att den inte är tillräckligt pålitlig för säkerhetsövervakning, om målet är att upptäcka intrång.

**B. Gradient Boosting (XGBoost/LightGBM)**

Snabbare och mer exakt än Random Forest för stora dataset.

Effektiv för obalanserad data (t.ex. få attacker men mycket normal trafik).

In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier()
model.fit(X_train, y_train)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.61


# **2. Oövervakad Inlärning (Unsupervised Learning)**
Används när: Ingen märkt data finns (t.ex. för nya/okända hot).

 **A. K-means Klustring**

Grupperar data i kluster och flaggar avvikelser.

Användbart för zero-day-exploits

In [2]:
# Importera nödvändiga bibliotek
from sklearn.cluster import KMeans  # För K-means klustringsalgoritmen
from sklearn.metrics import accuracy_score  # För att beräkna accuracy (används ej direkt här)
import numpy as np  # För numeriska operationer

# Skapa K-means modell med antal kluster som motsvarar antalet unika klasser i träningsdata
# Detta förutsätter att antalet kluster är känt och samma som antalet klasser
model = KMeans(n_clusters=len(np.unique(y_train)))

# Träna modellen på träningsdata (endast features, ingen användning av etiketter)
model.fit(X_train)

# Förutsäg kluster för testdata
y_pred = model.predict(X_test)

# För att beräkna accuracy måste vi matcha klustren med verkliga etiketter
# Eftersom K-means inte använder etiketter vid träning, finns det ingen garanti för att
# klusternas numrering matchar de verkliga klasserna

# Importera verktyg för optimal tilldelning av kluster till klasser
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix  # För att skapa en förvirringsmatris

# Skapa förvirringsmatris mellan sanna etiketter (y_test) och förutsagda kluster (y_pred)
conf_mat = confusion_matrix(y_test, y_pred)

# Hitta optimal matchning mellan kluster och klasser genom att maximera diagonalen i förvirringsmatrisen
# Den negativa matrisen används eftersom linear_sum_assignment minimerar total kostnad
row_ind, col_ind = linear_sum_assignment(-conf_mat)

# Beräkna accuracy som summan av rätt klassificerade sampel dividerat med totalt antal sampel
acc = conf_mat[row_ind, col_ind].sum() / len(y_test)

# Skriv ut accuracy avrundad till två decimaler
print("Accuracy:", round(acc, 2))
print("Förvirringsmatris: ", conf_mat)


Accuracy: 0.5
Förvirringsmatris:  [[101 111]
 [ 39  49]]


**B. Isolation Forest**

Specialdesignad för anomalidetektering.

Identifierar ovanliga mönster effektivt.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np

# Skapa Isolation Forest-modell
# contamination: Andel anomalier i data (kan sättas till 'auto' eller ett specifikt värde)
model = IsolationForest(contamination='auto', random_state=42)

# Träna modellen (använder bara X_train - ingen y_train eftersom det är oövervakat)
model.fit(X_train)

# Förutsägelser: Returnerar 1 för normala observationer, -1 för anomalier
y_pred = model.predict(X_test)

# Omvandla y_test till Isolation Forests format (1 för normala, -1 för anomalier)
# Förutsätter att y_test är 0/1 där 1 = normal och 0 = anomali (anpassa vid behov)
y_test_adapted = np.where(y_test == 0, -1, 1)

# Beräkna accuracy genom att jämföra förutsägelser med anpassade etiketter
accuracy = accuracy_score(y_test_adapted, y_pred)

print("Accuracy:", round(accuracy, 2))

Accuracy: 0.37


# **3. Djupinlärning (Deep Learning)**
**Används när: Data har komplexa mönster (t.ex. paketsekvenser).**

**A. Autoencoders**

Lär sig "normal" trafik och flaggar avvikelser.

Bra för tidssekvensdata (t.ex. nätverksflöden).

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix

# Förbered data
X = data.drop("label", axis=1)
y = data["label"]
X_scaled = MinMaxScaler().fit_transform(X) # Neurala nätverk fungerar bättre och snabbare när alla värden ligger inom ett liknande intervall — t.ex. 0 till 1.

# Dela upp data i tränings- och testdata
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42) # Om du inte anger random_state, får du olika uppdelningar varje gång du kör koden.


# Bygg en autoencoder (med 3 lager:  input layer, encoded-lager, decoded-lager (utmatningslager))
# Skapar ett inmatningslager med lika många neuroner som antal kolumner i datan
input_layer = Input(shape=(X_train.shape[1],))

# Komprimerar datan till 32 neuroner med ReLU-aktivering (kallas "encoding")
encoded = Dense(32, activation='relu')(input_layer)

# Återskapar (rekonstruerar) originaldatan från den komprimerade formen
decoded = Dense(X_train.shape[1], activation='sigmoid')(encoded)

# Skapar hela modellen: från input → encoded → decoded
autoencoder = Model(input_layer, decoded)

# Kompilerar modellen med:
# - "adam" som optimerare (hur modellen lär sig)
# - "mse" (Mean Squared Error) som förlustfunktion (mäter rekonstruktionsfel)
autoencoder.compile(optimizer='adam', loss='mse')


# Träna modellen
# batch_size=32 träna modellen på 32 rader åt gången istället för hela träningsdatan på en gång.
# verbose=0 Betyder: Visa inget i terminalen under träningen.
autoencoder.fit(X_train, X_train, epochs=20, batch_size=32, verbose=0)

# Beräkna rekonstruktionsfel (högre fel = avvikelse)
reconstruction = autoencoder.predict(X_test)
mse = np.mean(np.square(X_test - reconstruction), axis=1)

# Sätt en tröskel för avvikelser
# Om felet är högre än detta värde, är det ovanligt (kan vara en attack)
# 95:e percentilen av rekonstruktionsfelet (mse)
threshold = np.percentile(mse[y_test == 0], 95)
# Om mse > threshold → det är en attack (1). Annars → det är normalt (0)
y_pred = (mse > threshold).astype(int)

# Utvärdera modellen
accuracy = accuracy_score(y_test, y_pred)
print(f"Träffsäkerhet: {accuracy:.2%}")

# Förvirringsmatris
cm = confusion_matrix(y_test, y_pred)

# Skriv ut förvirringsmatrisen både som matris och individuella värden
print("\nFörvirringsmatris:")
print(cm)  # Skriver ut matrisen
print(f"\nNormal gissad som normal:     {cm[0][0]}")
print(f"Normal gissad som attack:     {cm[0][1]}")
print(f"Attack gissad som normal:     {cm[1][0]}")
print(f"Attack gissad som attack:     {cm[1][1]}")


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Träffsäkerhet: 69.00%

Förvirringsmatris:
[[201  11]
 [ 82   6]]

Normal gissad som normal:     201
Normal gissad som attack:     11
Attack gissad som normal:     82
Attack gissad som attack:     6
